# Lab: 2D Convolution on the Raspberry Pi
## Python → C → ARM64 Assembly

In this lab you run the **same operation**, a 2D convolution, written in three languages on
a Raspberry Pi 5, and then step by step in assembly. You will measure how long each version
takes and find out *why* the lower levels are faster.

You will not write a program from scratch. The code is ready and commented. Your job is to
**run it, read it, change one piece of assembly, and explain what you see** with your own
numbers.

### How to use this notebook

- Run a cell with **Shift + Enter**. Run the cells **in order**, from top to bottom.
- A cell marked **✏️** is yours: double-click it, write your answer, then Shift + Enter.
- Save often (**Ctrl + S**).

### What you hand in

1. This notebook, **exported as HTML** with all outputs and answers (see the last section).
2. The **Moodle quiz**, answered with the numbers from *your* runs.

---
## Part 0. Check the setup

The cell below checks that the system is 64-bit ARM (`aarch64`), builds the C and assembly
code with `make`, and runs a correctness test: every version must give exactly the same
result as Python on 13 image sizes.

**Expected:** `aarch64`, then the build, then a line ending in `all OK`.

In [ ]:
!uname -m
!cat /proc/device-tree/model; echo
!make
!python3 conv_bench.py --test

---
## Part 1. What is a convolution?

A **kernel** is a small grid of numbers, here 3 × 3. It slides over the image. At each
position, every pixel under the kernel is multiplied by the kernel number on top of it, and
the 9 products are added. That sum is one pixel of the output:

$$
\text{out}[y][x] = \sum_{k_y=0}^{2}\ \sum_{k_x=0}^{2}\ \text{in}[y+k_y][x+k_x] \cdot k[k_y][k_x]
$$

The kernel never hangs outside the image, so the output is 2 pixels smaller in each
direction. Image filters (blur, edge detection) and the neural networks that recognize
images are built on exactly this operation.

Run the cell below. It shows a small 6 × 8 image, the kernel, and how output pixel
**(0, 0)** is computed: the yellow 3 × 3 window is multiplied by the kernel and summed.

In [ ]:
from conv_demo import notebook_step, notebook_race

notebook_step(0, 0)

✏️ **Q1.** Using the image and kernel above, compute output pixel **out[2][3] by hand**.
Write the 9 products and the result.

*Hint: the window for out[2][3] starts at row 2, column 3 of the input.*

Then change the cell below to `notebook_step(2, 3)`, run it, and check your answer.

✏️ **Q1. Your 9 products and the result:**

*Double-click this cell and write here.*

In [ ]:
notebook_step(0, 0)   # change to (2, 3) to check your answer to Q1

✏️ **Q2.** The benchmark uses a **2400 × 2400** image. How many output pixels does it have,
and how many **multiply-adds** does the whole convolution need? Show the calculation.

✏️ **Q2. Your calculation:**

*Double-click this cell and write here.*

---
## Part 2. The same loops in three languages

### Python

Four nested loops: every output pixel, then every kernel position.

```python
for y in range(oh):                  # every output row ...
    for x in range(ow):              # ... every output column
        acc = 0
        for ky in range(3):          # every kernel row
            for kx in range(3):      # every kernel column
                acc += img[y + ky][x + kx] * k[ky][kx]
        out[y][x] = acc
```

Python does not run this directly on the CPU. The **interpreter** reads each operation
while the program runs: it looks up `img`, finds row `y + ky` in a list, finds the element,
checks that both values are numbers, creates a new integer object for the product, and so
on. Each `+` and `*` costs many steps.

### C (`src/conv.c`)

The same four loops. The difference is *when* the work happens: the compiler (`gcc -O2`)
translates them to machine instructions **once**, before the program runs.

```c
for (long y = 0; y < oh; y++)
    for (long x = 0; x < ow; x++) {
        int32_t acc = 0;
        for (long ky = 0; ky < 3; ky++)
            for (long kx = 0; kx < 3; kx++)
                acc += in[(y + ky) * w + (x + kx)] * k[ky * 3 + kx];
        out[y * ow + x] = acc;
    }
```

The image is one flat block of memory, row after row, so pixel `[r][c]` is at index
`r * w + c`.

### How Python calls C and assembly

`make` turns `src/conv.c` and `src/conv_asm.S` into **shared libraries** (`build/*.so`).
`conv_bench.py` loads them with Python's `ctypes` module and calls the functions as if they
were Python functions, passing the image as a block of memory. Python only measures the
time and checks the output; the work runs as native machine code.

### Assembly v1: brute force (`src/conv_asm.S`)

The first assembly version is a direct translation of the C loops. This is the innermost
loop, the one that runs once per multiply-add:

```asm
4:      add     x12, x8, x10            // y + ky
        mul     x12, x12, x4            // (y + ky) * w
        add     x13, x7, x11            // x + kx
        add     x12, x12, x13           // index into in[]
        ldr     w14, [x0, x12, lsl #2]  // pixel = in[index]
        mov     x15, #3
        mul     x15, x10, x15           // ky * 3
        add     x15, x15, x11           // index into k[]
        ldr     w16, [x2, x15, lsl #2]  // coef = k[index]
        mul     w14, w14, w16           // pixel * coef
        add     w9, w9, w14             // acc += pixel * coef
        add     x11, x11, #1            // kx++
        cmp     x11, #3
        b.lt    4b                      // repeat while kx < 3
```

`x` registers hold 64-bit values (addresses, counters); `w` registers are their low 32
bits (pixels). `ldr` loads from memory, `lsl #2` multiplies the index by 4 because each
pixel is 4 bytes.

✏️ **Q3.** In the v1 loop above:
1. How many instructions run for **one** multiply-add?
2. How many of them do the useful work (load the pixel, load the coefficient, multiply,
   add to `acc`), and how many only compute indexes or control the loop?

✏️ **Q3. Your answer:**

*Double-click this cell and write here.*

---
## Part 3. Measure all the versions

The cell below runs every version on the same 2400 × 2400 image of random numbers.
Python runs once (**about 10 seconds**); each C and assembly version runs 5 times and the
median is shown. Every output is compared with Python's: `OK` means identical.

Close other programs before running it.

In [ ]:
!python3 conv_bench.py

✏️ **Q4.** Copy your numbers into the table (two decimals).

| Version | Time (ms) | vs Python |
| --- | ---: | ---: |
| Python | | 1.0x |
| C (gcc -O2) | | |
| ASM v1 brute force | | |
| ASM v2 pointers | | |
| ASM v3 MAC (yours) | | |
| ASM v4 NEON | | |

*(Double-click to fill it in.)*

✏️ **Q5.** Where was the **biggest** jump in speed, and why?
Is a direct translation to assembly (v1) automatically faster than C compiled with
`-O2`? Look at your numbers.

✏️ **Q5. Your answer:**

*Double-click this cell and write here.*

---
## Part 4. Climbing the assembly ladder

Each assembly version changes **one idea** compared to the previous one, so each speed-up
has one cause. Open `src/conv_asm.S` to read the full code; every line is commented.

### v1 → v2: stop computing indexes inside the loop

v1 recomputes `(y + ky) * w + (x + kx)` with multiplications for every single multiply-add,
and reloads the kernel from memory every time. v2 instead:

- loads the 9 kernel numbers **once** into registers `w9`–`w17`;
- keeps **three pointers** (`x7`, `x8`, `x2`) on the three image rows under the kernel, and
  moves them 4 bytes to the right after each output pixel. No index arithmetic at all;
- writes out the 9 kernel positions one after another (no `ky`, `kx` loops).

`ldr w20, [x7], #4` loads a pixel **and** moves the pointer to the next one, in a single
instruction (*post-increment*). One output pixel of v2:

```asm
2:      ldr     w20, [x7], #4           // row y:   in[y][x], then p0 += 4
        mul     w19, w20, w9            //   acc  = pixel * k[0]
        ldr     w20, [x7]               //          in[y][x+1]
        mul     w20, w20, w10           //   product = pixel * k[1]
        add     w19, w19, w20           //   acc += product
        ...                             //   (the same for the other 7 positions)
        str     w19, [x1], #4           // *out++ = acc
        subs    x6, x6, #1              // columns left--
        b.ne    2b
```

### v2 → v3: the MAC instruction (your turn, Part 5)

In v2 every kernel position after the first needs `mul` and then `add`. ARM64 has a
**multiply-accumulate** (MAC) instruction that does both in one:

```asm
madd    Wd, Wn, Wm, Wa      // Wd = Wa + Wn * Wm
```

### v3 → v4: NEON, 4 pixels at once

The Raspberry Pi's CPU has **NEON** vector registers (`v0`–`v31`) of 128 bits, which hold
4 numbers of 32 bits each. One vector instruction works on all 4 at the same time:

```asm
ldr     q16, [x7]                   // load 4 pixels: in[y][x .. x+3]
mla     v20.4s, v16.4s, v0.s[1]     // 4 accumulators += 4 pixels * k[1]
```

v4 computes **4 output pixels per loop iteration**. The C compiler did not use these
instructions for `conv.c`; writing assembly by hand let us use them.

✏️ **Q6.** Count the instructions that run for **one output pixel** in v2 (loads,
multiplies, adds, the store, and the 2 loop instructions). How many is that per
multiply-add? Compare with your answer to Q3 for v1.

✏️ **Q6. Your answer:**

*Double-click this cell and write here.*

---
## Part 5. Hands-on: write v3 yourself

Open **[src/conv_student.S](src/conv_student.S)** (click the link, or use the Jupyter file
browser). It starts as an **exact copy of v2**, so right now the "ASM v3 MAC (yours)" row is
`OK` but no faster than v2.

**Your task:** the file has 8 places marked `TODO`, each a `mul` + `add` pair:

```asm
        mul     w20, w20, w10           // TODO 1: fuse this MUL ...
        add     w19, w19, w20           //         ... and this ADD into one MADD
```

Replace each pair with **one** `madd` instruction that gives the same result. Read the
comments at the top of the file first: they say which register holds what.

Save the file (Ctrl + S in the editor tab), come back here and run the cell below. It
rebuilds the code and runs the benchmark **without** Python (so it is quick); the reference
is then the C version.

- `NOT BUILT`: the assembler found an error. Read the message above the table, fix the
  line it names, and run again.
- `MISMATCH`: it builds but computes the wrong result. Check the order of the operands.
- To start over: `!git checkout src/conv_student.S`

In [ ]:
!make && python3 conv_bench.py --skip-python

The cell below prints the `madd` lines in your file, as proof of your solution in the
exported notebook. It must show 8 lines.

In [ ]:
!grep -nE "^\s+madd" src/conv_student.S

✏️ **Q7.** Compare the time of your v3 with v2. How many instructions per output pixel
does v3 run (use your count from Q6)? Did the speed change by the same proportion as the
instruction count? Propose one reason why it might not.

✏️ **Q7. Your answer:**

*Double-click this cell and write here.*

---
## Part 6. The race

Now see it live. Python, C and assembly v4 start the same 2400 × 2400 image **at the same
time**, each on its own CPU core of the Raspberry Pi. Each bar is the progress of that one
image. C and assembly finish almost at once; then they keep processing copies of the image
until Python finishes its first one.

Run the cell and watch the bars (about 15 seconds including the preparation).

In [ ]:
race = notebook_race()

✏️ **Q8.** While Python finished 1 image, how many images did C finish, and how many did
assembly v4? (The three versions share the memory during the race, so these numbers can
differ a little from the table in Part 3.)

✏️ **Q8. Your answer:**

*Double-click this cell and write here.*

---
## Part 7. Conclusions (200 words maximum)

Write your conclusions **in your own words** inside the quotes below, then run the cell to
count the words. Answer these points, using your numbers:

1. Where was the biggest jump in speed, and what causes it?
2. Is writing assembly by hand always faster than C? Use v1 vs C as evidence.
3. What did the pointers (v2), the MAC instruction (v3) and NEON (v4) each add?
4. Why does it matter to understand what code does at the lowest level?

In [ ]:
conclusions = """
Write your conclusions here.
"""

words = len(conclusions.split())
print(f"{words} words:", "OK" if words <= 200 else "too long, the maximum is 200")

---
## Hand in

1. Check that every cell has its output and every ✏️ cell has your answer.
2. Save (**Ctrl + S**).
3. Export: **File → Save and Export Notebook As… → HTML** (in the classic interface:
   **File → Download as → HTML**).
4. Upload the HTML file to Moodle and answer the **Moodle quiz** with the numbers from
   your runs.